# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing all dataset elements by their `@id` fields for transparency and reproducibility.

### Dataset Source
The dataset is described using a Croissant schema accessible at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Examine the available record sets, the fields they contain, and their unique `@id` identifiers.

Let's list all record sets and preview their field structure.

In [ ]:
# Retrieve available record sets by their @id
record_sets = dataset.record_sets
print(f"Available record sets (@id): {[r['@id'] for r in record_sets]}")

# Display fields and columns for each record set, referencing @id
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    print(f"Name: {rs.get('name', '<no name>')}")
    print(f"Description: {rs.get('description', '<no description>')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("Fields and columns by @id:")
    for f in fields:
        print(f"  Field @id: {f['@id']}")
        print(f"    Name: {f.get('name', '<no name>')}")
        columns = f.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        for c in columns:
            print(f"    Column @id: {c['@id']}")
            print(f"      Name: {c.get('name', '<no name>')}")

## 3. Data Extraction
We now extract the records from each record set using their `@id` fields.

This loads each record set as a DataFrame, retaining all fields and columns (referenced by their `@id`).

> **Note:** Use the record set and field @ids from the previous cell for precise referencing.

In [ ]:
# List all available record set @ids
record_sets_ids = [r['@id'] for r in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"\nLoaded DataFrame for Record Set @id {record_set_id} with shape {dataframes[record_set_id].shape}")
        print(f"Columns (by @id): {list(dataframes[record_set_id].columns)}")
        display(dataframes[record_set_id].head())
    else:
        print(f"\nRecord Set @id {record_set_id} contains no records.")

## 4. Exploratory Data Analysis (EDA)
Let's process and explore the main record set.

We'll:
- Filter rows based on a numeric field (referenced by its `@id`)
- Normalize it
- Group by another field (`@id`)

> *Adjust the `numeric_field_id` and `group_field_id` variables below to match the appropriate `@id`s from your dataset overview.*

In [ ]:
# Choose the main record set by @id (update if dataset contains more than one significant table)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Adjust if needed

    # Show available columns to pick relevant @id for numeric and group fields
    print(f"Fields available in Record Set @id {main_record_set_id}:\n", list(dataframes[main_record_set_id].columns))

    # For demonstration, choose plausible @id for numeric and group field
    # Suppose the main record set has the following columns:
    #   '@id:age', '@id:sex', '@id:tumor_location', '@id:msi_status', etc.
    # Let's pick '@id:age' as numeric and '@id:msi_status' as group if they exist
    numeric_field_id = None
    group_field_id = None
    for col in dataframes[main_record_set_id].columns:
        if 'age' in col.lower():
            numeric_field_id = col
        if 'msi' in col.lower():
            group_field_id = col
    # Fallbacks
    if not numeric_field_id:
        numeric_field_id = dataframes[main_record_set_id].select_dtypes(include='number').columns[0]
    if not group_field_id:
        group_field_id = dataframes[main_record_set_id].columns[1] if len(dataframes[main_record_set_id].columns) > 1 else dataframes[main_record_set_id].columns[0]

    print(f"\nUsing numeric field @id: {numeric_field_id}")
    print(f"Using group field @id: {group_field_id}")

    # Filter: rows where numeric_field_id > given threshold (example: age > 60)
    threshold = 60
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by group_field_id
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nAverage {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No DataFrames loaded; aborting EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and the mean by group. All field references use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id and group_field_id:
    # Histogram of the normalized numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[col_norm], kde=True, bins=20)
    plt.title(f"Distribution of normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel("Count")
    plt.show()

    # Barplot: mean of numeric_field grouped by group_field_id
    plt.figure(figsize=(7,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Average {numeric_field_id}")
    plt.xlabel(f"{group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded and explored the FAIR^2 clinical dataset using the Croissant schema via `mlcroissant`.
- All accesses were by `@id` to ensure robust, schema-driven data processing.
- We demonstrated field normalization and subgroup analysis for meaningful scientific inquiry.

This notebook can serve as a foundation for further clinical modeling and advanced data analyses using Croissant-compatible datasets.